## Expression prediction on our data from the lab

After a sanity check with scGPT, we confirmed it's prediction capabilities are much stronger than baseline using a revised pipeline. In this notebook we verift the transferability of that pipeline to our dataset, investigating differences in data that could cause mismatches.

### 1. Load sample data and verify representation

In [1]:
from pathlib import Path
import json
import sys

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.io import mmread

ROOT = Path.home() / "scgpt-cna"
MODEL_DIR = ROOT / "models/whole-human"

CASE_ID = "C3N-00148"
SAMPLE_ID = "C3N-00148-01"

SAMPLE_DIR = Path(
    "/home/akash1/projects/rrg-tperkins/dgladish/Projects/MSc/"
    "benchmarking_spatial_scale/GDC-data"
) / CASE_ID / SAMPLE_ID

RNA_DIR = (
    SAMPLE_DIR / "single_cell"
    / "dbe28c52-cdbf-477e-a728-9a61fd4a8139"
)
DATA_DIR = RNA_DIR / "qc_filtered_bc_feature_matrix"
COPYKAT_FILE = RNA_DIR / "copykatOUT/_copykat_prediction.txt"

print("Python:", sys.executable)
assert Path(sys.prefix).name == ".venv-scgpt", "Select the scGPT kernel."

# Accept either compressed or uncompressed 10x files.
def locate_file(directory, name):
    candidates = [
        p for p in [directory / name, directory / (name + ".gz")]
        if p.is_file()
    ]
    assert len(candidates) == 1, (
        f"Expected exactly one {name} or {name}.gz in {directory}; "
        f"found {candidates}"
    )
    return candidates[0]

MATRIX_FILE = locate_file(DATA_DIR, "matrix.mtx")
FEATURE_FILE = locate_file(DATA_DIR, "features.tsv")
BARCODE_FILE = locate_file(DATA_DIR, "barcodes.tsv")

assert COPYKAT_FILE.is_file(), COPYKAT_FILE
assert (MODEL_DIR / "vocab.json").is_file()

print("Sample:", SAMPLE_ID)
print("Matrix:", MATRIX_FILE)

Python: /home/akash1/scgpt-cna/.venv-scgpt/bin/python
Sample: C3N-00148-01
Matrix: /home/akash1/projects/rrg-tperkins/dgladish/Projects/MSc/benchmarking_spatial_scale/GDC-data/C3N-00148/C3N-00148-01/single_cell/dbe28c52-cdbf-477e-a728-9a61fd4a8139/qc_filtered_bc_feature_matrix/matrix.mtx


In [2]:
# load raw matricies

features = pd.read_csv(
    FEATURE_FILE, sep="\t", header=None, dtype=str, na_filter=False
)
barcodes = pd.read_csv(
    BARCODE_FILE, sep="\t", header=None, dtype=str, na_filter=False
)

assert features.shape[1] == 3, features.shape
assert barcodes.shape[1] == 1, barcodes.shape

features.columns = ["gene_id", "gene_symbol", "feature_type"]
assert features["feature_type"].eq("Gene Expression").all()
assert features["gene_id"].is_unique
assert barcodes[0].is_unique
assert features["gene_id"].ne("").all()
assert barcodes[0].ne("").all()

# Stored matrix: genes × cells.
# AnnData convention: cells × genes.
print("Reading sparse counts...", flush=True)
counts = mmread(MATRIX_FILE).T.tocsr()
counts.sum_duplicates()
counts.eliminate_zeros()
counts.sort_indices()

assert sparse.issparse(counts)
assert counts.shape == (len(barcodes), len(features))
assert np.isfinite(counts.data).all()
assert (counts.data >= 0).all()
assert np.allclose(
    counts.data, np.rint(counts.data), rtol=0, atol=1e-6
), "Expected raw integer-valued counts."

obs = pd.DataFrame(
    index=pd.Index(barcodes[0].to_numpy(), name="barcode")
)
obs["case_id"] = CASE_ID
obs["sample_id"] = SAMPLE_ID

var = features.copy()
var.index = pd.Index(var["gene_id"].to_numpy(), name="feature_id")

adata_lab = ad.AnnData(X=counts, obs=obs, var=var)
adata_lab.layers["counts"] = adata_lab.X.copy()

adata_lab.obs["total_counts"] = np.asarray(
    counts.sum(axis=1)
).ravel()
adata_lab.obs["n_genes_detected"] = counts.getnnz(axis=1)

print(adata_lab)
print(f"Nonzero entries: {counts.nnz:,}")
display(adata_lab.var.head())
display(
    adata_lab.obs[["total_counts", "n_genes_detected"]].describe()
)

Reading sparse counts...
AnnData object with n_obs × n_vars = 7598 × 60660
    obs: 'case_id', 'sample_id', 'total_counts', 'n_genes_detected'
    var: 'gene_id', 'gene_symbol', 'feature_type'
    layers: 'counts'
Nonzero entries: 12,599,909


,gene_id,gene_symbol,feature_type
feature_id,,,
ENSG00000223972.5,ENSG00000223972.5,DDX11L1,Gene Expression
ENSG00000227232.5,ENSG00000227232.5,WASH7P,Gene Expression
ENSG00000278267.1,ENSG00000278267.1,MIR6859-1,Gene Expression
ENSG00000243485.5,ENSG00000243485.5,MIR1302-2HG,Gene Expression
ENSG00000284332.1,ENSG00000284332.1,MIR1302-2,Gene Expression


,total_counts,n_genes_detected
count,7598.000000,7598.000000
mean,3166.449329,1658.319163
std,1676.129470,656.013332
min,500.000000,64.000000
25%,2025.500000,1196.250000
50%,2902.000000,1616.000000
75%,3963.750000,2057.000000
max,24279.000000,6980.000000


In [3]:
print(barcodes.head())
print(features.head())


                    0
0  AAACCCAAGAAGTGTT-1
1  AAACCCAAGCCTGTCG-1
2  AAACCCAAGCGCTGCT-1
3  AAACCCAAGTTCCGGC-1
4  AAACCCACAAGCGCTC-1
             gene_id  gene_symbol     feature_type
0  ENSG00000223972.5      DDX11L1  Gene Expression
1  ENSG00000227232.5       WASH7P  Gene Expression
2  ENSG00000278267.1    MIR6859-1  Gene Expression
3  ENSG00000243485.5  MIR1302-2HG  Gene Expression
4  ENSG00000284332.1    MIR1302-2  Gene Expression


In [4]:
# align copyKAT predictions by parcode
copykat = pd.read_csv(COPYKAT_FILE, sep="\t", dtype=str)

required = {"cell.names", "copykat.pred"}
assert required.issubset(copykat.columns), copykat.columns.tolist()
assert copykat["cell.names"].notna().all()
assert copykat["cell.names"].is_unique
assert copykat["copykat.pred"].notna().all()

predictions = copykat.set_index("cell.names")["copykat.pred"]

matched = adata_lab.obs_names.isin(predictions.index)
extra_barcodes = predictions.index.difference(adata_lab.obs_names)

adata_lab.obs["copykat_prediction"] = predictions.reindex(
    adata_lab.obs_names
).to_numpy()

print(f"Matched cells: {matched.sum():,}/{adata_lab.n_obs:,}")
print("CopyKAT barcodes absent from expression:", len(extra_barcodes))
if len(extra_barcodes):
    print("Examples:", extra_barcodes[:5].tolist())

print("\nCopyKAT categories; NaN means no matching output row:")
display(
    adata_lab.obs["copykat_prediction"]
    .value_counts(dropna=False)
    .rename("n_cells")
)

Matched cells: 7,584/7,598
CopyKAT barcodes absent from expression: 0

CopyKAT categories; NaN means no matching output row:


copykat_prediction
aneuploid      4106
diploid        2100
not.defined    1378
NaN              14
Name: n_cells, dtype: int64

In [5]:
vocab = json.loads((MODEL_DIR / "vocab.json").read_text())

symbols = adata_lab.var["gene_symbol"]
in_vocab = symbols.isin(vocab)
keep_for_model = in_vocab & ~symbols.duplicated(keep="first")

adata_lab.var["in_scgpt_vocab"] = in_vocab.to_numpy()
adata_lab.var["keep_for_model"] = keep_for_model.to_numpy()

gene_totals = np.asarray(
    adata_lab.layers["counts"].sum(axis=0)
).ravel()
all_counts = float(gene_totals.sum())
assert all_counts > 0

coverage_rows = []
for label, mask in [
    ("All features", np.ones(adata_lab.n_vars, dtype=bool)),
    ("Vocabulary matches", in_vocab.to_numpy()),
    ("After keeping first symbol occurrence", keep_for_model.to_numpy()),
]:
    retained_counts = float(gene_totals[mask].sum())
    coverage_rows.append({
        "stage": label,
        "n_features": int(mask.sum()),
        "total_counts": retained_counts,
        "percent_counts_retained": 100 * retained_counts / all_counts,
    })

coverage_lab = pd.DataFrame(coverage_rows)
display(coverage_lab.round(3))

print(
    "Repeated symbol occurrences beyond the first:",
    int(symbols.duplicated(keep="first").sum()),
)
print(
    "Matched features removed for duplicate symbols:",
    int((in_vocab & ~keep_for_model).sum()),
)

# Per-cell coverage can reveal losses hidden by the overall average.
retained_per_cell = np.asarray(
    adata_lab.layers["counts"][:, keep_for_model.to_numpy()].sum(axis=1)
).ravel()

totals = adata_lab.obs["total_counts"].to_numpy()
fractions = np.full(adata_lab.n_obs, np.nan)
np.divide(retained_per_cell, totals, out=fractions, where=totals > 0)
adata_lab.obs["model_count_fraction"] = fractions

print("\nPer-cell fraction of counts retained:")
display(adata_lab.obs["model_count_fraction"].describe())

# Identify the most expressed features that fail vocabulary matching.
unmatched = adata_lab.var.loc[
    ~in_vocab, ["gene_id", "gene_symbol"]
].copy()
unmatched["total_counts"] = gene_totals[~in_vocab.to_numpy()]

print("\nMost expressed unmatched features:")
display(
    unmatched.sort_values("total_counts", ascending=False).head(10)
)

,stage,n_features,total_counts,percent_counts_retained
0,All features,60660,24058682.0,100.000
1,Vocabulary matches,39839,22524177.0,93.622
2,After keeping first symbol occurrence,39800,22524114.0,93.622


Repeated symbol occurrences beyond the first: 1233
Matched features removed for duplicate symbols: 39

Per-cell fraction of counts retained:


count    7598.000000
mean        0.936876
std         0.021378
min         0.861687
25%         0.920641
50%         0.933591
75%         0.953378
max         1.000000
Name: model_count_fraction, dtype: float64


Most expressed unmatched features:


,gene_id,gene_symbol,total_counts
feature_id,,,
ENSG00000256947.1,ENSG00000256947.1,AP002518.2,39541
ENSG00000223786.1,ENSG00000223786.1,AL357507.1,20813
ENSG00000287733.1,ENSG00000287733.1,AC083862.2,19851
ENSG00000247199.5,ENSG00000247199.5,AC011346.1,18326
ENSG00000236039.3,ENSG00000236039.3,AC019117.1,12685
ENSG00000272243.6,ENSG00000272243.6,AL356277.3,11009
ENSG00000233005.2,ENSG00000233005.2,AC018742.1,10585
ENSG00000230258.6,ENSG00000230258.6,AC005208.1,10010
ENSG00000275443.2,ENSG00000275443.2,AC012409.2,9726


### 2. Compare lab data to census data

In [6]:
# load census and compare gene names
CENSUS_FILE = (
    ROOT / "sanity_check/data/census_proximal_tubule_256/reference_raw.h5ad"
)
assert CENSUS_FILE.is_file()

adata_census = ad.read_h5ad(CENSUS_FILE)
print("Lab:", adata_lab.shape)
print("Census:", adata_census.shape)

# Build a lookup using Census's stable Ensembl IDs.
census_annotations = pd.DataFrame({
    "ensembl_base": (
        adata_census.var["feature_id"].astype(str)
        .str.replace(r"\.\d+$", "", regex=True).to_numpy()
    ),
    "census_symbol": (
        adata_census.var["feature_name"].astype(str).to_numpy()
    ),
})

# Exclude ambiguous IDs rather than silently choosing a mapping.
ambiguous = census_annotations["ensembl_base"].duplicated(keep=False)
print("Census rows with ambiguous base IDs:", int(ambiguous.sum()))

symbol_lookup = (
    census_annotations.loc[~ambiguous]
    .set_index("ensembl_base")["census_symbol"]
)

gene_comparison = pd.DataFrame({
    "gene_id": adata_lab.var["gene_id"].to_numpy(),
    "lab_symbol": adata_lab.var["gene_symbol"].astype(str).to_numpy(),
    "total_counts": np.asarray(
        adata_lab.layers["counts"].sum(axis=0)
    ).ravel(),
})
gene_comparison["ensembl_base"] = (
    gene_comparison["gene_id"].str.replace(r"\.\d+$", "", regex=True)
)
gene_comparison["census_symbol"] = (
    gene_comparison["ensembl_base"].map(symbol_lookup)
)

gene_comparison["lab_symbol_in_vocab"] = (
    gene_comparison["lab_symbol"].isin(vocab)
)
gene_comparison["census_symbol_in_vocab"] = (
    gene_comparison["census_symbol"].isin(vocab)
)
gene_comparison["different_symbol"] = (
    gene_comparison["census_symbol"].notna()
    & gene_comparison["lab_symbol"].ne(gene_comparison["census_symbol"])
)

potentially_recovered = (
    ~gene_comparison["lab_symbol_in_vocab"]
    & gene_comparison["census_symbol_in_vocab"]
)

print("Lab features with an unambiguous Census ID match:",
      int(gene_comparison["census_symbol"].notna().sum()))
print("Shared-ID features with different symbols:",
      int(gene_comparison["different_symbol"].sum()))
print("Currently unmatched features with a recognized Census symbol:",
      int(potentially_recovered.sum()))
print(
    "Percent of all lab counts in those potentially recoverable features:",
    round(
        100 * gene_comparison.loc[
            potentially_recovered, "total_counts"
        ].sum() / gene_comparison["total_counts"].sum(),
        3,
    ),
)

display(
    gene_comparison.loc[potentially_recovered]
    .sort_values("total_counts", ascending=False)
    .head(15)
)

Lab: (7598, 60660)
Census: (256, 60664)
Census rows with ambiguous base IDs: 0
Lab features with an unambiguous Census ID match: 60561
Shared-ID features with different symbols: 21170
Currently unmatched features with a recognized Census symbol: 20765
Percent of all lab counts in those potentially recoverable features: 6.363


,gene_id,lab_symbol,total_counts,ensembl_base,census_symbol,lab_symbol_in_vocab,census_symbol_in_vocab,different_symbol
34603,ENSG00000256947.1,AP002518.2,39541,ENSG00000256947,RP11-64D24.2,False,True,True
20178,ENSG00000223786.1,AL357507.1,20813,ENSG00000223786,RP11-554D15.1,False,True,True
24007,ENSG00000287733.1,AC083862.2,19851,ENSG00000287733,RP11-134L10.2,False,True,True
17851,ENSG00000247199.5,AC011346.1,18326,ENSG00000247199,RP11-373N22.3,False,True,True
21868,ENSG00000236039.3,AC019117.1,12685,ENSG00000236039,LINC02889,False,True,True
20180,ENSG00000272243.6,AL356277.3,11009,ENSG00000272243,RP11-554D15.3,False,True,True
5799,ENSG00000233005.2,AC018742.1,10585,ENSG00000233005,AC067959.1,False,True,True
49206,ENSG00000230258.6,AC005208.1,10010,ENSG00000230258,AC002539.1,False,True,True
43912,ENSG00000275443.2,AC012409.2,9726,ENSG00000275443,RP11-759A24.3,False,True,True
59782,ENSG00000284377.1,AL589669.1,9471,ENSG00000284377,RP11-226B15.1,False,True,True


In [7]:
# load historical preprocessing function
HISTORICAL = (
    ROOT / "sanity_check/reference_4068d67/scgpt/preprocess.py"
)
assert HISTORICAL.is_file()

import importlib.util

spec = importlib.util.spec_from_file_location(
    "transfer_historical_preprocess", HISTORICAL
)
historical = importlib.util.module_from_spec(spec)
spec.loader.exec_module(historical)
assert historical._digitize.__defaults__ == ("one",)

config = json.loads((MODEL_DIR / "args.json").read_text())
assert config["n_bins"] == 51

def profile_dataset(adata, symbol_column):
    raw = sparse.csr_matrix(adata.layers["counts"], copy=True)
    raw.sum_duplicates()
    raw.eliminate_zeros()

    names = adata.var[symbol_column].astype(str)
    keep = names.isin(vocab) & ~names.duplicated(keep="first")
    selected = raw[:, keep.to_numpy()].tocsr()

    total = np.asarray(raw.sum(axis=1)).ravel()
    detected = raw.getnnz(axis=1)
    model_total = np.asarray(selected.sum(axis=1)).ravel()
    model_detected = selected.getnnz(axis=1)

    assert (total > 0).all()
    assert (detected > 0).all()
    assert (model_detected > 0).all()

    # Count genes with exactly one detected molecule/count.
    ones = raw.copy()
    ones.data = (ones.data == 1).astype(np.int32)
    n_ones = np.asarray(ones.sum(axis=1)).ravel()

    records = []
    for i in range(adata.n_obs):
        values = selected.getrow(i).data.astype(np.float32)
        binned = historical.binning(values.copy(), n_bins=51)
        labels, frequencies = np.unique(binned, return_counts=True)
        count_one = values == 1

        records.append({
            "model_fraction_count_one": float(count_one.mean()),
            "count_one_bin": (
                int(binned[count_one][0]) if count_one.any() else np.nan
            ),
            "occupied_bins": len(labels),
            "dominant_bin_fraction": float(frequencies.max() / len(binned)),
        })

        if (i + 1) % 2000 == 0:
            print(f"Profiled {i + 1:,}/{adata.n_obs:,} cells", flush=True)

    profile = pd.DataFrame(records, index=adata.obs_names)
    profile["total_counts"] = total
    profile["detected_genes"] = detected
    profile["model_detected_genes"] = model_detected
    profile["model_count_fraction"] = model_total / total
    profile["raw_fraction_count_one"] = n_ones / detected

    return profile, selected, keep

lab_profile, lab_model_counts, lab_keep = profile_dataset(
    adata_lab, "gene_symbol"
)
census_profile, census_model_counts, census_keep = profile_dataset(
    adata_census, "feature_name"
)

# Carry annotations alongside the lab measurements.
lab_profile["copykat_prediction"] = adata_lab.obs["copykat_prediction"]

metrics = [
    "total_counts",
    "detected_genes",
    "model_detected_genes",
    "model_count_fraction",
    "raw_fraction_count_one",
    "model_fraction_count_one",
    "count_one_bin",
    "occupied_bins",
    "dominant_bin_fraction",
]

comparison_rows = []
for metric in metrics:
    row = {"metric": metric}
    for name, profile in [
        ("lab", lab_profile), ("census", census_profile)
    ]:
        values = profile[metric].dropna()
        row[f"{name}_q25"] = values.quantile(0.25)
        row[f"{name}_median"] = values.median()
        row[f"{name}_q75"] = values.quantile(0.75)
    comparison_rows.append(row)

distribution_comparison = pd.DataFrame(comparison_rows).set_index("metric")
display(distribution_comparison.round(3))

for name, profile in [("Lab", lab_profile), ("Census", census_profile)]:
    print(
        f"{name}: "
        f"{100 * profile['model_detected_genes'].gt(1199).mean():.1f}% "
        "of cells exceed 1,199 matched detected genes."
    )

/home/akash1/scgpt-cna/.venv-scgpt/lib/python3.10/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/home/akash1/scgpt-cna/.venv-scgpt/lib/python3.10/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


Profiled 2,000/7,598 cells
Profiled 4,000/7,598 cells
Profiled 6,000/7,598 cells


,lab_q25,lab_median,lab_q75,census_q25,census_median,census_q75
metric,,,,,,
total_counts,2025.500,2902.000,3963.750,772.000,1571.500,3399.750
detected_genes,1196.250,1616.000,2057.000,472.750,786.500,1294.250
model_detected_genes,1099.000,1481.000,1874.000,472.750,786.500,1294.250
model_count_fraction,0.921,0.934,0.953,1.000,1.000,1.000
raw_fraction_count_one,0.705,0.747,0.790,0.645,0.697,0.767
model_fraction_count_one,0.700,0.744,0.788,0.645,0.697,0.767
count_one_bin,35.000,37.000,39.000,32.000,35.000,38.000
occupied_bins,5.000,6.000,6.000,6.000,7.000,9.000
dominant_bin_fraction,0.700,0.744,0.788,0.645,0.697,0.767


Lab: 68.4% of cells exceed 1,199 matched detected genes.
Census: 30.1% of cells exceed 1,199 matched detected genes.


In [8]:
grouped = lab_profile.copy()
grouped["copykat_group"] = grouped["copykat_prediction"].fillna(
    "missing_from_output"
)

group_summary = grouped.groupby("copykat_group").agg(
    n_cells=("total_counts", "size"),
    median_counts=("total_counts", "median"),
    median_detected_genes=("detected_genes", "median"),
    median_model_detected_genes=("model_detected_genes", "median"),
    median_count_coverage=("model_count_fraction", "median"),
    median_fraction_count_one=("model_fraction_count_one", "median"),
    median_count_one_bin=("count_one_bin", "median"),
    median_occupied_bins=("occupied_bins", "median"),
)

display(group_summary.round(3))

,n_cells,median_counts,median_detected_genes,median_model_detected_genes,median_count_coverage,median_fraction_count_one,median_count_one_bin,median_occupied_bins
copykat_group,,,,,,,,
aneuploid,4106,3367.0,1818.0,1647.0,0.924,0.735,37.0,6.0
diploid,2100,2916.0,1590.0,1495.5,0.956,0.725,36.0,6.0
missing_from_output,14,1556.5,136.5,135.5,0.999,0.798,39.5,8.5
not.defined,1378,1456.0,876.5,819.0,0.950,0.812,40.0,5.0


In [9]:
# Confirm gene_comparison is still in the original feature order.
assert np.array_equal(
    gene_comparison["gene_id"].to_numpy(),
    adata_lab.var["gene_id"].to_numpy(),
)

original_symbols = adata_lab.var["gene_symbol"].astype(str)

census_symbols = pd.Series(
    gene_comparison["census_symbol"].to_numpy(),
    index=adata_lab.var_names,
)

# Prefer the ID-matched Census symbol when it is recognized.
# Otherwise retain the original lab symbol.
use_census = census_symbols.isin(vocab)
model_symbols = original_symbols.where(~use_census, census_symbols)

adata_lab.var["model_symbol"] = model_symbols
adata_lab.var["model_symbol_source"] = np.where(
    use_census, "Ensembl-ID-matched Census annotation", "original lab annotation"
)

new_in_vocab = model_symbols.isin(vocab)
new_keep = new_in_vocab & ~model_symbols.duplicated(keep="first")

adata_lab.var["keep_for_model_harmonized"] = new_keep.to_numpy()

gene_totals = np.asarray(
    adata_lab.layers["counts"].sum(axis=0)
).ravel()

rows = []
for name, mask in [
    ("Original matching, after duplicate handling", lab_keep.to_numpy()),
    ("Harmonized matching, before duplicate handling", new_in_vocab.to_numpy()),
    ("Harmonized matching, after duplicate handling", new_keep.to_numpy()),
]:
    rows.append({
        "selection": name,
        "n_features": int(mask.sum()),
        "percent_counts_retained": (
            100 * gene_totals[mask].sum() / gene_totals.sum()
        ),
    })

display(pd.DataFrame(rows).round(4))

# Inspect all recognized features involved in a repeated model symbol.
duplicate_mask = (
    new_in_vocab & model_symbols.duplicated(keep=False)
)
duplicates = adata_lab.var.loc[
    duplicate_mask, ["gene_id", "gene_symbol", "model_symbol"]
].copy()
duplicates["total_counts"] = gene_totals[duplicate_mask.to_numpy()]

print("Recognized features involved in duplicate model symbols:",
      len(duplicates))
print("Counts discarded by keeping the first occurrence:",
      float(gene_totals[(new_in_vocab & ~new_keep).to_numpy()].sum()))

display(
    duplicates.sort_values(
        ["model_symbol", "total_counts"], ascending=[True, False]
    ).head(30)
)

,selection,n_features,percent_counts_retained
0,"Original matching, after duplicate handling",39800,93.6216
1,"Harmonized matching, before duplicate handling",60604,99.9850
2,"Harmonized matching, after duplicate handling",60566,99.9848


Recognized features involved in duplicate model symbols: 76
Counts discarded by keeping the first occurrence: 63.0


,gene_id,gene_symbol,model_symbol,total_counts
feature_id,,,,
ENSG00000182584.6,ENSG00000182584.6,ACTL10,ACTL10,1
ENSG00000288649.1,ENSG00000288649.1,ACTL10,ACTL10,0
ENSG00000197976.12,ENSG00000197976.12,AKAP17A,AKAP17A,657
ENSG00000197976.12_PAR_Y,ENSG00000197976.12_PAR_Y,AKAP17A,AKAP17A,0
ENSG00000237801.6,ENSG00000237801.6,AMD1P2,AMD1P2,1
ENSG00000237801.6_PAR_Y,ENSG00000237801.6_PAR_Y,AMD1P2,AMD1P2,0
ENSG00000196433.13,ENSG00000196433.13,ASMT,ASMT,10
ENSG00000196433.13_PAR_Y,ENSG00000196433.13_PAR_Y,ASMT,ASMT,0
ENSG00000169093.16,ENSG00000169093.16,ASMTL,ASMTL,414


In [10]:
# Preserve the original matching results for comparison.
lab_profile_original = lab_profile.copy()
lab_model_counts_original = lab_model_counts

# Reuse our profiling function, now matching through harmonized symbols.
(
    lab_profile_harmonized,
    lab_model_counts_harmonized,
    lab_keep_harmonized,
) = profile_dataset(adata_lab, "model_symbol")

lab_profile_harmonized["copykat_prediction"] = (
    adata_lab.obs["copykat_prediction"]
)

assert np.array_equal(
    lab_keep_harmonized.to_numpy(),
    adata_lab.var["keep_for_model_harmonized"].to_numpy(),
)

# Integer vocabulary IDs, aligned with the model matrix's columns.
lab_gene_ids = np.array([
    vocab[s]
    for s in adata_lab.var.loc[lab_keep_harmonized, "model_symbol"]
], dtype=np.int64)

assert lab_model_counts_harmonized.shape == (
    adata_lab.n_obs, len(lab_gene_ids)
)
assert len(np.unique(lab_gene_ids)) == len(lab_gene_ids)

profiles = {
    "Lab: original names": lab_profile_original,
    "Lab: harmonized names": lab_profile_harmonized,
    "Census": census_profile,
}

comparison_metrics = [
    "model_detected_genes",
    "model_count_fraction",
    "model_fraction_count_one",
    "count_one_bin",
    "occupied_bins",
    "dominant_bin_fraction",
]

median_comparison = pd.DataFrame({
    name: profile[comparison_metrics].median()
    for name, profile in profiles.items()
})
display(median_comparison.round(4))

print("Final lab model matrix:", lab_model_counts_harmonized.shape)
print(
    "Harmonized lab cells exceeding 1,199 detected model genes:",
    f"{100 * lab_profile_harmonized['model_detected_genes'].gt(1199).mean():.1f}%",
)

Profiled 2,000/7,598 cells
Profiled 4,000/7,598 cells
Profiled 6,000/7,598 cells


,Lab: original names,Lab: harmonized names,Census
model_detected_genes,1481.0000,1616.0000,786.5000
model_count_fraction,0.9336,1.0000,1.0000
model_fraction_count_one,0.7437,0.7473,0.6968
count_one_bin,37.0000,37.0000,35.0000
occupied_bins,6.0000,6.0000,7.0000
dominant_bin_fraction,0.7437,0.7473,0.6968


Final lab model matrix: (7598, 60566)
Harmonized lab cells exceeding 1,199 detected model genes: 74.7%


### 3. Run small execution check

In [11]:
import torch

x_lab = lab_model_counts_harmonized
assert x_lab.shape[1] == len(lab_gene_ids)
assert x_lab.shape[0] >= 256

rng = np.random.default_rng(42)
order = rng.permutation(x_lab.shape[0])

lab_reference_indices = order[:128]
lab_evaluation_indices = order[128:256]

assert not set(lab_reference_indices) & set(lab_evaluation_indices)

# Eight evaluation cells spanning the number of detected model genes.
detected = x_lab.getnnz(axis=1)
ordered_eval = lab_evaluation_indices[
    np.argsort(detected[lab_evaluation_indices])
]
positions = np.linspace(0, len(ordered_eval) - 1, 8, dtype=int)
lab_smoke_indices = ordered_eval[positions]

# Baselines use historical bins in reference cells only.
ref_sum = np.zeros(x_lab.shape[1], dtype=np.float64)
ref_support = np.zeros(x_lab.shape[1], dtype=np.int64)

for i in lab_reference_indices:
    row = x_lab.getrow(int(i))
    bins = historical.binning(
        row.data.astype(np.float32).copy(), n_bins=51
    )
    ref_sum[row.indices] += bins
    ref_support[row.indices] += 1

lab_constant = float(ref_sum.sum() / ref_support.sum())
lab_gene_mean = np.full(x_lab.shape[1], lab_constant)
np.divide(
    ref_sum, ref_support,
    out=lab_gene_mean, where=ref_support > 0,
)

print("Reference cells:", len(lab_reference_indices))
print("Reserved evaluation cells:", len(lab_evaluation_indices))
print("Reference constant:", round(lab_constant, 3))

display(
    adata_lab.obs.iloc[lab_smoke_indices][[
        "copykat_prediction", "total_counts", "n_genes_detected"
    ]]
)

Reference cells: 128
Reserved evaluation cells: 128
Reference constant: 38.665


,copykat_prediction,total_counts,n_genes_detected
barcode,,,
CGGAATTCACCGTGGT-1,not.defined,568,494
ATCGGCGTCATCACCC-1,diploid,1838,973
GGAACCCGTGGATCAG-1,aneuploid,2463,1375
CACAGATCAAACCACT-1,diploid,2923,1614
TAACACGAGCTTCTAG-1,aneuploid,3407,1844
TGGTGATGTGGTCAAG-1,aneuploid,3957,2099
CCTAACCTCCGAGTGC-1,aneuploid,4766,2344
GACTGATTCGGAATGG-1,diploid,7771,3524


In [12]:
# Load historical collator, explicitly binding historical binning.
collator_path = HISTORICAL.parent / "data_collator.py"
module_name = "scgpt._lab_historical_collator"

spec = importlib.util.spec_from_file_location(module_name, collator_path)
collator_module = importlib.util.module_from_spec(spec)
sys.modules[module_name] = collator_module
spec.loader.exec_module(collator_module)

collator_module.binning = historical.binning
assert collator_module.binning is historical.binning
assert config["n_bins"] == 51

pad_id = vocab[config.get("pad_token", "<pad>")]
cls_id = vocab["<cls>"]
pad_value = config.get("pad_value", -2)

lab_collator = collator_module.DataCollator(
    do_padding=True,
    pad_token_id=pad_id,
    pad_value=pad_value,
    do_mlm=True,
    do_binning=True,
    mlm_probability=0.25,
    mask_value=config.get("mask_value", -1),
    max_length=config["max_seq_len"],
    sampling=True,
    keep_first_n_tokens=1,
    data_style="both",
)

examples = []
expected_bins = []

for i in lab_smoke_indices:
    row = x_lab.getrow(int(i))
    ids = lab_gene_ids[row.indices]
    values = row.data.astype(np.float32)

    bins = historical.binning(values.copy(), n_bins=51)
    expected_bins.append(dict(zip(ids.tolist(), bins.tolist())))

    examples.append({
        "genes": torch.tensor(
            np.concatenate(([cls_id], ids)), dtype=torch.long
        ),
        "expressions": torch.tensor(
            np.concatenate(([pad_value], values)), dtype=torch.float32
        ),
    })

torch.manual_seed(42)
np.random.seed(42)
lab_smoke_batch = lab_collator(examples)

for j in range(len(lab_smoke_indices)):
    vg = lab_smoke_batch["pcpt_gene"][j]
    vv = lab_smoke_batch["pcpt_expr"][j]
    qg = lab_smoke_batch["gen_gene"][j]
    qt = lab_smoke_batch["gen_expr_target"][j]

    visible = vg.ne(pad_id)
    query = qg.ne(pad_id)

    visible_ids = vg[visible].tolist()
    visible_values = vv[visible].tolist()
    query_ids = qg[query].tolist()
    targets = qt[query].tolist()

    assert visible_ids[0] == cls_id
    assert visible_values[0] == pad_value
    assert cls_id not in query_ids
    assert len(query_ids) > 0
    assert set(visible_ids[1:]).isdisjoint(query_ids)

    for gene, value in zip(visible_ids[1:], visible_values[1:]):
        assert value == expected_bins[j][gene]
    for gene, value in zip(query_ids, targets):
        assert value == expected_bins[j][gene]

print("Visible/query separation and expression targets verified.")
for name, tensor in lab_smoke_batch.items():
    print(name, tuple(tensor.shape))

Visible/query separation and expression targets verified.
pcpt_gene (8, 901)
pcpt_expr (8, 901)
gen_gene (8, 299)
gen_expr_target (8, 299)


In [13]:
from time import perf_counter

helper_file = ROOT / "notebooks/scgpt_session/scgpt_session.py"
assert helper_file.is_file(), helper_file

module_name = "lab_scgpt_inference_helper"
spec = importlib.util.spec_from_file_location(module_name, helper_file)
lab_session = importlib.util.module_from_spec(spec)
sys.modules[module_name] = lab_session
spec.loader.exec_module(lab_session)

torch.set_num_threads(4)
lab_model, loaded_vocab, loaded_config = lab_session.load_model(MODEL_DIR)
assert loaded_config == config
assert all(loaded_vocab[s] == int(vocab[s]) for s in vocab)

lab_model = lab_model.cpu().eval()
token_to_column = {
    int(token): column for column, token in enumerate(lab_gene_ids)
}

results = []
lab_smoke_predictions = {}
start = perf_counter()

with torch.inference_mode():
    for j, i in enumerate(lab_smoke_indices):
        visible = lab_smoke_batch["pcpt_gene"][j].ne(pad_id)
        query = lab_smoke_batch["gen_gene"][j].ne(pad_id)

        known_ids = lab_smoke_batch["pcpt_gene"][j][visible][None, :]
        known_values = lab_smoke_batch["pcpt_expr"][j][visible][None, :]
        query_ids = lab_smoke_batch["gen_gene"][j][query][None, :]

        target = (
            lab_smoke_batch["gen_expr_target"][j][query]
            .numpy().astype(np.float64)
        )

        # The model receives no target expression values.
        output = lab_model(known_ids, known_values, query_ids)

        columns = np.array([
            token_to_column[int(g)] for g in query_ids[0].tolist()
        ])

        predictions = {
            "GEP": output["GEP"][0].numpy(),
            "MVC": output["MVC"][0].numpy(),
            "gene_mean": lab_gene_mean[columns],
            "reference_constant": np.full(target.shape, lab_constant),
            "visible_mean": np.full(
                target.shape, float(known_values[0, 1:].mean())
            ),
        }

        barcode = str(adata_lab.obs_names[int(i)])
        lab_smoke_predictions[barcode] = {
            "query_ids": query_ids[0].numpy().copy(),
            "target": target.copy(),
            "predictions": {
                name: pred.copy() for name, pred in predictions.items()
            },
        }

        for method, pred in predictions.items():
            pred = np.asarray(pred, dtype=np.float64)
            assert pred.shape == target.shape
            assert np.isfinite(pred).all()

            results.append({
                "barcode": barcode,
                "copykat_group": adata_lab.obs.iloc[int(i)][
                    "copykat_prediction"
                ],
                "model_detected_genes": int(detected[int(i)]),
                "method": method,
                "n_queries": len(target),
                "target_mean": target.mean(),
                "prediction_mean": pred.mean(),
                "prediction_std": pred.std(),
                "MAE": np.abs(pred - target).mean(),
                "MSE": np.square(pred - target).mean(),
            })

        print(f"Completed {j + 1}/{len(lab_smoke_indices)}", flush=True)

lab_smoke_metrics = pd.DataFrame(results)

display(lab_smoke_metrics.round(3))
display(
    lab_smoke_metrics.pivot(
        index="barcode", columns="method", values="MSE"
    ).round(3)
)
print(f"Elapsed: {perf_counter() - start:.1f} seconds")

Strict load: 163 checkpoint tensors; no omitted tensors.
Completed 1/8
Completed 2/8
Completed 3/8
Completed 4/8
Completed 5/8
Completed 6/8
Completed 7/8
Completed 8/8


,barcode,copykat_group,model_detected_genes,method,n_queries,target_mean,prediction_mean,prediction_std,MAE,MSE
0,CGGAATTCACCGTGGT-1,not.defined,494,GEP,123,46.244,46.653,0.154,0.767,0.791
1,CGGAATTCACCGTGGT-1,not.defined,494,MVC,123,46.244,46.249,0.251,0.431,0.617
2,CGGAATTCACCGTGGT-1,not.defined,494,gene_mean,123,46.244,38.872,2.304,7.382,59.678
3,CGGAATTCACCGTGGT-1,not.defined,494,reference_constant,123,46.244,38.665,0.000,7.579,58.117
4,CGGAATTCACCGTGGT-1,not.defined,494,visible_mean,123,46.244,46.197,0.000,0.409,0.674
5,ATCGGCGTCATCACCC-1,diploid,973,GEP,243,41.473,41.622,1.191,2.190,8.118
6,ATCGGCGTCATCACCC-1,diploid,973,MVC,243,41.473,41.426,1.153,2.074,7.914
7,ATCGGCGTCATCACCC-1,diploid,973,gene_mean,243,41.473,38.746,2.165,3.312,18.768
8,ATCGGCGTCATCACCC-1,diploid,973,reference_constant,243,41.473,38.665,0.000,2.809,16.541
9,ATCGGCGTCATCACCC-1,diploid,973,visible_mean,243,41.473,41.412,0.000,2.316,8.656


method,GEP,MVC,gene_mean,reference_constant,visible_mean
barcode,,,,,
ATCGGCGTCATCACCC-1,8.118,7.914,18.768,16.541,8.656
CACAGATCAAACCACT-1,11.710,11.626,12.715,14.433,14.556
CCTAACCTCCGAGTGC-1,22.605,21.849,23.613,27.898,25.568
CGGAATTCACCGTGGT-1,0.791,0.617,59.678,58.117,0.674
GACTGATTCGGAATGG-1,25.704,25.624,36.778,39.091,35.792
GGAACCCGTGGATCAG-1,9.005,9.264,11.687,13.639,10.133
TAACACGAGCTTCTAG-1,13.876,13.875,13.836,16.778,16.247
TGGTGATGTGGTCAAG-1,15.932,15.942,15.864,19.524,19.527


Elapsed: 1.5 seconds


In [14]:
# run on 128 cells now
def make_lab_example(i):
    row = x_lab.getrow(int(i))
    return {
        "genes": torch.tensor(
            np.concatenate(([cls_id], lab_gene_ids[row.indices])),
            dtype=torch.long,
        ),
        "expressions": torch.tensor(
            np.concatenate(([pad_value], row.data.astype(np.float32))),
            dtype=torch.float32,
        ),
    }

torch.manual_seed(42)
np.random.seed(42)

lab_evaluation_batch = lab_collator([
    make_lab_example(i) for i in lab_evaluation_indices
])

evaluation_rows = []
lab_model.eval()
start = perf_counter()

with torch.inference_mode():
    for j, i in enumerate(lab_evaluation_indices):
        i = int(i)
        visible = lab_evaluation_batch["pcpt_gene"][j].ne(pad_id)
        query = lab_evaluation_batch["gen_gene"][j].ne(pad_id)

        known_ids = lab_evaluation_batch["pcpt_gene"][j][visible][None, :]
        known_values = lab_evaluation_batch["pcpt_expr"][j][visible][None, :]
        query_ids = lab_evaluation_batch["gen_gene"][j][query][None, :]

        target = (
            lab_evaluation_batch["gen_expr_target"][j][query]
            .numpy().astype(np.float64)
        )

        assert known_ids[0, 0].item() == cls_id
        assert known_values[0, 0].item() == pad_value
        assert len(target) > 0
        assert set(known_ids[0, 1:].tolist()).isdisjoint(
            query_ids[0].tolist()
        )

        correct = lab_model(known_ids, known_values, query_ids)

        # Preserve gene IDs and expression distribution, but break their pairing.
        generator = torch.Generator().manual_seed(10000 + i)
        permutation = torch.randperm(
            known_values.shape[1] - 1, generator=generator
        )
        shuffled_values = known_values.clone()
        shuffled_values[0, 1:] = known_values[0, 1:][permutation]

        shuffled = lab_model(known_ids, shuffled_values, query_ids)

        columns = np.array([
            token_to_column[int(g)] for g in query_ids[0].tolist()
        ])

        predictions = {
            "GEP": correct["GEP"][0].numpy(),
            "MVC": correct["MVC"][0].numpy(),
            "GEP_shuffled": shuffled["GEP"][0].numpy(),
            "MVC_shuffled": shuffled["MVC"][0].numpy(),
            "gene_mean": lab_gene_mean[columns],
            "reference_constant": np.full(target.shape, lab_constant),
            "visible_mean": np.full(
                target.shape, float(known_values[0, 1:].mean())
            ),
        }

        annotations = adata_lab.obs.iloc[i]
        group = annotations["copykat_prediction"]
        group = "missing_from_output" if pd.isna(group) else str(group)

        for method, pred in predictions.items():
            pred = np.asarray(pred, dtype=np.float64)
            assert pred.shape == target.shape
            assert np.isfinite(pred).all()

            evaluation_rows.append({
                "cell_index": i,
                "barcode": str(adata_lab.obs_names[i]),
                "copykat_group": group,
                "total_counts": float(annotations["total_counts"]),
                "model_detected_genes": int(detected[i]),
                "method": method,
                "n_queries": len(target),
                "target_mean": float(target.mean()),
                "target_std": float(target.std()),
                "prediction_mean": float(pred.mean()),
                "prediction_std": float(pred.std()),
                "gene_baseline_support_fraction":
                    float((ref_support[columns] > 0).mean()),
                "MAE": float(np.abs(pred - target).mean()),
                "MSE": float(np.square(pred - target).mean()),
            })

        if (j + 1) % 16 == 0:
            print(f"Completed {j + 1}/{len(lab_evaluation_indices)}", flush=True)

lab_evaluation_metrics = pd.DataFrame(evaluation_rows)

# Each cell receives equal weight.
summary = lab_evaluation_metrics.groupby("method").agg(
    n_cells=("cell_index", "nunique"),
    mean_MSE=("MSE", "mean"),
    median_MSE=("MSE", "median"),
    mean_MAE=("MAE", "mean"),
).sort_values("mean_MSE")

paired = lab_evaluation_metrics.pivot(
    index="cell_index", columns="method", values="MSE"
)

comparisons = []
for decoder in ["GEP", "MVC"]:
    for baseline in [
        "gene_mean", "reference_constant",
        "visible_mean", f"{decoder}_shuffled",
    ]:
        improvement = paired[baseline] - paired[decoder]
        comparisons.append({
            "decoder": decoder,
            "comparison": baseline,
            "mean_MSE_improvement": improvement.mean(),
            "median_MSE_improvement": improvement.median(),
            "cells_decoder_wins": int((improvement > 0).sum()),
            "n_cells": len(improvement),
        })

display(summary.round(3))
display(pd.DataFrame(comparisons).round(3))

# Descriptive subgroup results; CopyKAT groups are not cell-type labels.
display(
    lab_evaluation_metrics.groupby(["copykat_group", "method"]).agg(
        n_cells=("cell_index", "nunique"),
        mean_MSE=("MSE", "mean"),
        mean_MAE=("MAE", "mean"),
    ).round(3)
)

output_dir = ROOT / "sanity_check/results"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / f"{SAMPLE_ID}_harmonized_25pct_seed42.csv"
lab_evaluation_metrics.to_csv(output_file, index=False)

print(f"Elapsed: {perf_counter() - start:.1f} seconds")
print("Saved:", output_file)

Completed 16/128
Completed 32/128
Completed 48/128
Completed 64/128
Completed 80/128
Completed 96/128
Completed 112/128
Completed 128/128


,n_cells,mean_MSE,median_MSE,mean_MAE
method,,,,
GEP,128,14.515,14.234,2.914
MVC,128,14.652,14.364,2.881
GEP_shuffled,128,15.342,14.754,3.284
MVC_shuffled,128,15.395,14.809,3.211
visible_mean,128,18.042,17.322,3.483
gene_mean,128,21.381,18.503,3.790
reference_constant,128,23.477,20.185,4.014


,decoder,comparison,mean_MSE_improvement,median_MSE_improvement,cells_decoder_wins,n_cells
0,GEP,gene_mean,6.866,3.129,111,128
1,GEP,reference_constant,8.961,5.385,128,128
2,GEP,visible_mean,3.526,2.719,126,128
3,GEP,GEP_shuffled,0.827,0.416,116,128
4,MVC,gene_mean,6.729,3.011,109,128
5,MVC,reference_constant,8.825,5.203,128,128
6,MVC,visible_mean,3.390,2.685,127,128
7,MVC,MVC_shuffled,0.744,0.336,110,128


n_cells  mean_MSE  mean_MAE
copykat_group method                                         
aneuploid     GEP                      73    15.947     3.145
              GEP_shuffled             73    16.461     3.480
              MVC                      73    15.982     3.117
              MVC_shuffled             73    16.493     3.404
              gene_mean                73    18.618     3.474
              reference_constant       73    21.574     3.740
              visible_mean             73    18.911     3.677
diploid       GEP                      35    16.539     3.136
              GEP_shuffled             35    18.325     3.684
              MVC                      35    16.962     3.135
              MVC_shuffled             35    18.509     3.651
              gene_mean                35    25.214     4.220
              reference_constant       35    27.318     4.540
              visible_mean             35    22.534     3.997
not.defined   GEP                      20     5.749     1.683
              GEP_shuffled             20     6.040     1.869
              MVC                      20     5.752     1.579
              MVC_shuffled             20     5.939     1.738
              gene_mean                20    24.758     4.192
              reference_constant       20    23.698     4.095
              visible_mean             20     7.007     1.877

Elapsed: 36.0 seconds
Saved: /home/akash1/scgpt-cna/sanity_check/results/C3N-00148-01_harmonized_25pct_seed42.csv


### 4. Evaluate results

In [15]:
# One row per evaluation cell, using GEP for this first diagnostic.
diagnostics = (
    lab_evaluation_metrics
    .loc[lab_evaluation_metrics["method"].eq("GEP")]
    .set_index("cell_index")
    .copy()
)

mse_by_method = lab_evaluation_metrics.pivot(
    index="cell_index", columns="method", values="MSE"
)

for baseline in ["gene_mean", "visible_mean", "GEP_shuffled"]:
    diagnostics[f"gain_vs_{baseline}"] = (
        mse_by_method[baseline] - mse_by_method["GEP"]
    )

# Match the previously calculated QC measurements by barcode.
qc = lab_profile_harmonized.reindex(diagnostics["barcode"].tolist())
assert qc["model_fraction_count_one"].notna().all()

diagnostics["fraction_count_one"] = (
    qc["model_fraction_count_one"].to_numpy()
)
diagnostics["occupied_bins"] = qc["occupied_bins"].to_numpy()
diagnostics["target_variance"] = diagnostics["target_std"] ** 2

# Separate overall level error from errors in gene-to-gene variation.
diagnostics["squared_mean_bias"] = (
    diagnostics["prediction_mean"] - diagnostics["target_mean"]
) ** 2
diagnostics["centered_MSE"] = (
    diagnostics["MSE"] - diagnostics["squared_mean_bias"]
)

display(
    diagnostics.groupby("copykat_group").agg(
        n_cells=("barcode", "size"),
        median_detected_genes=("model_detected_genes", "median"),
        median_fraction_count_one=("fraction_count_one", "median"),
        median_target_std=("target_std", "median"),
        median_prediction_std=("prediction_std", "median"),
        mean_MSE=("MSE", "mean"),
        mean_squared_bias=("squared_mean_bias", "mean"),
        mean_centered_MSE=("centered_MSE", "mean"),
        mean_gain_vs_gene_mean=("gain_vs_gene_mean", "mean"),
        mean_gain_vs_visible_mean=("gain_vs_visible_mean", "mean"),
        mean_context_gain=("gain_vs_GEP_shuffled", "mean"),
    ).round(3)
)

# Descriptive rank correlations, not causal effects or significance tests.
features = [
    "total_counts", "model_detected_genes",
    "fraction_count_one", "target_std",
]
outcomes = [
    "MSE", "gain_vs_gene_mean",
    "gain_vs_visible_mean", "gain_vs_GEP_shuffled",
]

display(
    diagnostics[features + outcomes]
    .corr(method="spearman")
    .loc[features, outcomes]
    .round(3)
)

,n_cells,median_detected_genes,median_fraction_count_one,median_target_std,median_prediction_std,mean_MSE,mean_squared_bias,mean_centered_MSE,mean_gain_vs_gene_mean,mean_gain_vs_visible_mean,mean_context_gain
copykat_group,,,,,,,,,,,
aneuploid,73,1871.0,0.727,4.233,1.788,15.947,0.046,15.901,2.672,2.964,0.514
diploid,35,1703.0,0.713,4.919,2.310,16.539,0.047,16.492,8.675,5.995,1.786
not.defined,20,774.0,0.839,2.282,1.098,5.749,0.065,5.684,19.009,1.258,0.291


,MSE,gain_vs_gene_mean,gain_vs_visible_mean,gain_vs_GEP_shuffled
total_counts,0.872,-0.446,0.694,0.496
model_detected_genes,0.870,-0.440,0.695,0.492
fraction_count_one,-0.973,0.340,-0.832,-0.602
target_std,0.984,-0.335,0.867,0.638


In [16]:
def evaluate_lab_mask(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    # make_lab_example creates fresh tensors for the mutating collator.
    batch = lab_collator([
        make_lab_example(i) for i in lab_evaluation_indices
    ])

    rows = []
    lab_model.eval()

    with torch.inference_mode():
        for j, i in enumerate(lab_evaluation_indices):
            i = int(i)

            visible = batch["pcpt_gene"][j].ne(pad_id)
            query = batch["gen_gene"][j].ne(pad_id)

            known_ids = batch["pcpt_gene"][j][visible][None, :]
            known_values = batch["pcpt_expr"][j][visible][None, :]
            query_ids = batch["gen_gene"][j][query][None, :]
            target = (
                batch["gen_expr_target"][j][query]
                .numpy().astype(np.float64)
            )

            assert known_ids[0, 0].item() == cls_id
            assert known_values[0, 0].item() == pad_value
            assert len(target) > 0
            assert set(known_ids[0, 1:].tolist()).isdisjoint(
                query_ids[0].tolist()
            )

            correct = lab_model(known_ids, known_values, query_ids)

            generator = torch.Generator().manual_seed(seed * 100000 + i)
            permutation = torch.randperm(
                known_values.shape[1] - 1, generator=generator
            )
            shuffled_values = known_values.clone()
            shuffled_values[0, 1:] = known_values[0, 1:][permutation]

            shuffled = lab_model(known_ids, shuffled_values, query_ids)

            columns = np.array([
                token_to_column[int(g)] for g in query_ids[0].tolist()
            ])

            predictions = {
                "GEP": correct["GEP"][0].numpy(),
                "MVC": correct["MVC"][0].numpy(),
                "GEP_shuffled": shuffled["GEP"][0].numpy(),
                "MVC_shuffled": shuffled["MVC"][0].numpy(),
                "gene_mean": lab_gene_mean[columns],
                "reference_constant": np.full(target.shape, lab_constant),
                "visible_mean": np.full(
                    target.shape, float(known_values[0, 1:].mean())
                ),
            }

            annotations = adata_lab.obs.iloc[i]
            group = annotations["copykat_prediction"]
            group = "missing_from_output" if pd.isna(group) else str(group)

            for method, pred in predictions.items():
                pred = np.asarray(pred, dtype=np.float64)
                assert pred.shape == target.shape
                assert np.isfinite(pred).all()

                rows.append({
                    "mask_seed": seed,
                    "cell_index": i,
                    "barcode": str(adata_lab.obs_names[i]),
                    "copykat_group": group,
                    "total_counts": float(annotations["total_counts"]),
                    "model_detected_genes": int(detected[i]),
                    "method": method,
                    "n_queries": len(target),
                    "target_mean": float(target.mean()),
                    "target_std": float(target.std()),
                    "prediction_mean": float(pred.mean()),
                    "prediction_std": float(pred.std()),
                    "MAE": float(np.abs(pred - target).mean()),
                    "MSE": float(np.square(pred - target).mean()),
                })

            if (j + 1) % 32 == 0:
                print(
                    f"Seed {seed}: {j + 1}/{len(lab_evaluation_indices)} cells",
                    flush=True,
                )

    return pd.DataFrame(rows)


# Preserve the original seed-42 run and add three new runs.
runs = [lab_evaluation_metrics.assign(mask_seed=42).copy()]

start = perf_counter()
for seed in [43, 44, 45]:
    runs.append(evaluate_lab_mask(seed))

lab_repeated_metrics = pd.concat(runs, ignore_index=True)

assert not lab_repeated_metrics.duplicated(
    ["mask_seed", "cell_index", "method"]
).any()

print(f"Completed additional runs in {perf_counter() - start:.1f} seconds")

Seed 43: 32/128 cells
Seed 43: 64/128 cells
Seed 43: 96/128 cells
Seed 43: 128/128 cells
Seed 44: 32/128 cells
Seed 44: 64/128 cells
Seed 44: 96/128 cells
Seed 44: 128/128 cells
Seed 45: 32/128 cells
Seed 45: 64/128 cells
Seed 45: 96/128 cells
Seed 45: 128/128 cells
Completed additional runs in 105.5 seconds


In [17]:
print("Mean MSE by mask seed:")
display(
    lab_repeated_metrics.groupby(["mask_seed", "method"])["MSE"]
    .mean()
    .unstack("mask_seed")
    .round(3)
)

# Average repeated measurements within each cell.
cell_averages = (
    lab_repeated_metrics.groupby(["cell_index", "method"])
    .agg(
        MSE=("MSE", "mean"),
        MAE=("MAE", "mean"),
        n_masks=("mask_seed", "nunique"),
    )
    .reset_index()
)
assert cell_averages["n_masks"].eq(4).all()

print("Performance after averaging masks within each cell:")
display(
    cell_averages.groupby("method").agg(
        n_cells=("cell_index", "nunique"),
        mean_MSE=("MSE", "mean"),
        median_cell_MSE=("MSE", "median"),
        mean_MAE=("MAE", "mean"),
    ).sort_values("mean_MSE").round(3)
)

wide = lab_repeated_metrics.pivot(
    index=["mask_seed", "cell_index"], columns="method", values="MSE"
)

stability_rows = []
for decoder in ["GEP", "MVC"]:
    for baseline in ["gene_mean", "visible_mean", f"{decoder}_shuffled"]:
        gains = wide[baseline] - wide[decoder]
        average_gain = gains.groupby(level="cell_index").mean()
        wins = gains.gt(0).groupby(level="cell_index").sum()

        stability_rows.append({
            "decoder": decoder,
            "comparison": baseline,
            "mean_MSE_gain": average_gain.mean(),
            "median_cell_gain": average_gain.median(),
            "cells_win_on_average": int(average_gain.gt(0).sum()),
            "cells_win_all_4_masks": int(wins.eq(4).sum()),
            "n_cells": len(average_gain),
        })

display(pd.DataFrame(stability_rows).round(3))

output_dir = ROOT / "sanity_check/results"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / f"{SAMPLE_ID}_harmonized_25pct_repeated_masks.csv"
lab_repeated_metrics.to_csv(output_file, index=False)
print("Saved:", output_file)

Mean MSE by mask seed:


mask_seed,42,43,44,45
method,,,,
GEP,14.515,14.298,14.400,14.587
GEP_shuffled,15.342,15.280,15.264,15.460
MVC,14.652,14.412,14.548,14.695
MVC_shuffled,15.395,15.296,15.317,15.500
gene_mean,21.381,21.073,21.309,21.325
reference_constant,23.477,23.126,23.367,23.425
visible_mean,18.042,17.777,17.894,18.009


Performance after averaging masks within each cell:


,n_cells,mean_MSE,median_cell_MSE,mean_MAE
method,,,,
GEP,128,14.450,14.094,2.908
MVC,128,14.577,14.297,2.872
GEP_shuffled,128,15.336,14.861,3.284
MVC_shuffled,128,15.377,14.946,3.210
visible_mean,128,17.931,17.163,3.479
gene_mean,128,21.272,18.399,3.780
reference_constant,128,23.349,19.484,4.004


,decoder,comparison,mean_MSE_gain,median_cell_gain,cells_win_on_average,cells_win_all_4_masks,n_cells
0,GEP,gene_mean,6.822,3.137,110,92,128
1,GEP,visible_mean,3.481,2.600,127,125,128
2,GEP,GEP_shuffled,0.887,0.462,122,92,128
3,MVC,gene_mean,6.695,3.011,109,89,128
4,MVC,visible_mean,3.354,2.552,128,125,128
5,MVC,MVC_shuffled,0.800,0.377,124,83,128


Saved: /home/akash1/scgpt-cna/sanity_check/results/C3N-00148-01_harmonized_25pct_repeated_masks.csv


### 5. Evaluate transferability for all samples in the lab data